## Prerequisites
* numpy
* termcolor
* tqdm
* prettytable

Note that in Jupyter, you can install a dependency via pip:
```
%pip install numpy termcolor tqdm prettytable
```

pip install ipywidgetspip install ipywidgets

In [24]:
pip install ipywidgets


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from functools import partial
from itertools import combinations
from collections import defaultdict, Counter
import random
import copy
import time

import numpy as np
from termcolor import cprint
from tqdm.notebook import tqdm
from prettytable import PrettyTable

import kille
from utils import run_tournament, play_game, Strategy

## Heuristic implementation
Each team will develop a function `student_eval(board: List[List[str]], stone: str) -> float`, which will return a number describing how promising the input board looks for player `stone` (assuming it is player `stone`'s turn). 

For example, `student_eval` would return 
* A very large positive number if the game state on `board` is clearly winning (or already won) for player `stone`
* A very large negative number if the game state on `board` is clearly losing (or already lost) for player `stone`
* 0 if the board appears even

In [83]:
# =============================================================================
# All of your helper functions go here!
# =============================================================================
'''HELPER FUNCTIONS
The following are our helper functions for our student_eval function.
'''
def other_stone(stone):
    """
    Return the opposite player's stone in a Gomoku game.
    Inputs:
        - stone, a string representing the current player's stone ("X" or "O")
    Returns: the string representing the other player's stone
    """
    return "X" if stone == "O" else "O"

def consecutive_k(row, k, stone):
    """
    Count the number of times a given stone appears in k consecutive positions in a row.
    Inputs:
        - row, a list of strings representing stones in a row
        - k, an integer representing the length of the consecutive sequence to check for
        - stone, a string representing the player's stone

    Returns: the number of k-length sequences in the row that match [stone] * k
    """
    desired_row = [stone] * k
    return sum(desired_row == row[i:i+k] for i in range(len(row) - k + 1))

def get_downdiag(board, row_idx, col_idx):
    """
    Extract the downward (top-left to bottom-right) diagonal starting at a given position.
    Inputs:
        - board, a list of lists representing the game board
        - row_idx, the starting row index
        - col_idx, the starting column index
    Returns: a list of strings representing the stones along the downward diagonal
    """
    return [board[row_idx + i][col_idx + i] for i in range(min(len(board) - row_idx, len(board) - col_idx))]

def get_updiag(board, row_idx, col_idx):
    """
    Extract the upward (bottom-left to top-right) diagonal starting at a given position.
    Inputs:
        - board, a list of lists representing the game board
        - row_idx, the starting row index
        - col_idx, the starting column index
    Returns: a list of strings representing the stones along the upward diagonal
    """
    return [board[row_idx - i][col_idx + i] for i in range(min(row_idx + 1, len(board) - col_idx))]

def iter_seqs(board):
    """
    Extracts all the rows, columns, and diagonals of the Gomoku board
    Note: only returns diagonal seqs of length at least 5
    Input:
        - board, a list of lists representing the game board
    Yields each row, column, and diagonal as a List
    """
    for row in board:
        yield list(row)

    for col_idx in range(len(board)):
        yield [row[col_idx] for row in board]

    for row_idx in range(len(board)):
        d = get_updiag(board, row_idx, 0)
        if len(d) >= 5:
            yield d
        d = get_downdiag(board, row_idx, 0)
        if len(d) >= 5:
            yield d

    for col_idx in range(len(board[0])):
        d = get_updiag(board, len(board) - 1, col_idx)
        if len(d) >= 5:
            yield d
        d = get_downdiag(board, 0, col_idx)
        if len(d) >= 5:
            yield d

''' --OPENED_ENDS FUNCTION--
The opened_ends function serves as a helper function that checks for open ends of k consecutive stones (where k can be [2,3,4]), either from the opponent's stones or ours, if so, it will return the number of open end sequences of stones found in the row given by the get_iter function, which iterates all diagonals, verticals, and rows to lists of rows. The inputs of this function are the row given by get_iter, k, and the stone to detect consecutive k of. The function will return a count of the amount of open ends it finds in the board.
--INPUT--
The input of the opened_ends function are:
    row - a list of strings that represents a row of the board
    k - an integer that represents the size of the window
    stone - a string that represents the stone of the current player
--OUTPUT--
Returns:
    count - an integer that represents the number of open ends it finds in the board]
'''
def opened_ends(row, k, stone):
    count = 0
    row_length = len(row)
    window_size = k + 2
    for i in range(row_length - window_size + 1):
        window = row[i : i + window_size]
        if window[0] == '-' and window[-1] == '-':
            middle = window[1:-1]
            if consecutive_k(middle, k, stone) == 1:
                count += 1
    return count

## Old version of p_blocked_ends, where we used front and back windows
#def p_blocked_ends(row,k, stone):
#    count = 0
#    row_length = len(row)
#    if row_length < 5:
#        return 0
#    front_window = row[0:5]
#    if front_window[0] == stone:
#        front_empty = 0
#        for j in front_window:
#            if j == '-':
#                front_empty += 1
#        if front_empty == (5 - k):
#            if consecutive_k(front_window, k , stone) == 1:
#                count += 1
#    back_window = row[-5:]
#    if back_window[-1] == stone:
#        back_empty = 0
#        for e in back_window:
#            if e == '-':
#                back_empty += 1
#        if back_empty == (5 - k):
#            if consecutive_k(back_window, k , stone) == 1:
#                count += 1
#    return count

'''--P_BLOCKED_ENDS FUNCTION--
The p_blocked_ends function works by creating a sliding window of size k (where k can be of the size [2,3,4]), and checks if the window is occupied by the stone. The function's purpose is to count the number of consecutive k where a consecutive sequence of k stones is blocked by an opponent's stone on one of its sides, either by a wall of the board or to an opponent's stone.
--INPUT--
The input of the p_blocked_ends function are:
    row - a list of strings that represents a row of the board
    k - an integer that represents the size of the window
    stone - a string that represents the stone of the current player

--OUTPUT--
Returns:
    count - an integer that represents the number of consecutive k where a consecutive sequence of k stones is blocked by an opponent's stone on one of its sides, either by a wall of the board or to an opponent's stone.
'''
def p_blocked_ends(row, k, stone):
    count = 0
    row_length = len(row)
    opponent = other_stone(stone)
    ## Creating a sliding window
    for i in range(row_length - k - 1):
        window = row[i:i+k]
        full_k = True
        for index in window:
            if index != stone:
                full_k = False
        if full_k:
            left_index = i-1
            right_index = i+k
            determine_left_edge = left_index < 0
            determine_right_edge = right_index >= row_length
            ## Sliding window, looks at both sides of the sliding window
            blocked_left = determine_left_edge or row[left_index] == opponent
            blocked_right = determine_right_edge or row[right_index] == opponent
            open_left = (not determine_left_edge) and row[left_index] == '-'
            open_right = (not determine_right_edge) and row[right_index] == '-'
            if (blocked_left and open_right) or (blocked_right and open_left):
                count += 1
    return count

# Detect split interval, this function instead of detecting the sides of a sequence of numbers, it will detect the inside of the sliding window, so that it can detect split combinations of k, let's say there's a row from get_iter that returns a sequence k that is XX_XX, the opened ends and blocked end functions would solely detect combinations of two separately given that it depends of a consecutive number

'''--DETECT_SPLIT_K--
The detect_split_k function checks for inner split combinations of k (where k can be [2,3,4]), it detects the splits by a gap of one between one another of the same type of stone.
--INPUT--
The input of the p_blocked_ends function
    row - a list of strings that represents a row of the board
    k - an integer that represents the size of the window
    stone - a string that represents the stone of the current played
--OUTPUT--
Returns:
    count - the number of inner split k combinations found of the given stone on the board
'''
def detect_split_k(row, k, stone):
    count = 0
    row_length = len(row)
    combination_k = [stone] * k
    for i in range(row_length - 4):
        window = row[i:i+5]
        if window.count(stone) == k and window.count('-') == 5-k:
            has_consecutive = False
            for e in range(5-k+1):
                if window[e:e+k] == combination_k:
                    has_consecutive = True
                    break
            if not has_consecutive:
                count += 1
    return count

'''--GOMU_GOMU_NO_MI--
The gomu_gomu_no_Mi function evaluates whether the first turn has been made or not
--INPUT--
The input of the detect_1st_turn function is:
    board - a list of lists hat contains strings representing the stones of the board
    stone - a string that represents the next player to make a move
--OUTPUT--
Return:
    A float that puts in positive all values to place the first stone at,
    depending if the first turn has been made or not
'''
def gomu_gomu_no_Mi(board, stone):
    float_value = 0.0
    opponent = other_stone(stone)
    board_len = len(board)
    stone_positions = []
    opponent_count = 0
    '''Detects any stones on the board,
    summing to the stone_count if one is found and defaults to 1
    if there's no stone. Sums opponent stones if present,
    else defaults to 0.'''
    for r in range(board_len):
        for c in range(board_len):
            cell = board[r][c]
            if cell == stone:
                stone_positions.append((r, c))
            elif cell == opponent:
                opponent_count += 1
    stone_count = len(stone_positions)
    '''PLACING FIRST STONE - Detects if the first stone is placed
    on the board and if there are no opponent stones present.
    The stone count defaults to 1 as
    it is the first turn even though there's no stone placed,
    a feature from the play_game function.'''
    if stone_count == 1 and opponent_count == 0:
        row_idx, col_idx = stone_positions[0]
        cells_to_target = [(6, 8), (8, 6), (6, 6), (8, 8)]
        if (row_idx, col_idx) in cells_to_target:
            float_value = 5000.0
        return float_value
    '''PLACING SECOND STONE - Detects if the second stone is placed
    on the board and if there are no opponent stones present'''
    if stone_count == 2:
        first, second = stone_positions
        if first == (8, 6) or second == (8, 6):
            other = second if first == (8, 6) else first
            cells_to_target = [(9, 5), (9, 7), (7, 5)]
            if other in cells_to_target:
                float_value = 5000.0
                return float_value
        if first == (8, 8) or second == (8, 8):
            other = second if first == (8, 8) else first
            cells_to_target = [(9, 7), (9, 9), (7, 9)]
            if other in cells_to_target:
                float_value = 5000.0
                return float_value
        if first == (6, 8) or second == (6, 8):
            other = second if first == (6, 8) else first
            cells_to_target = [(5, 9), (5, 7), (7, 9)]
            if other in cells_to_target:
                float_value = 5000.0
                return float_value
        if first == (6, 6) or second == (6, 6):
            other = second if first == (6, 6) else first
            cells_to_target = [(5, 5), (5, 7), (7, 5)]
            if other in cells_to_target:
                float_value = 5000.0
                return float_value
    return float_value

'''Try 1: Tried to get an area of the board to place x randomly depending on the turn'''
    ## FIRST TURN: stone has placed exactly one piece and the opponent has
    ## placed none -- reward landing in the center box. Spec was rows/cols
    ## [5,10] on a 15x15 board, scaled here to whatever size board we're
    ## actually on.
#    if stone_count == 1 and opponent_count == 0:
#        lo = round(board_len * 1/4)
#        hi = round(board_len * 1/3)
#        if lo <= row_idx <= hi and lo <= col_idx <= hi:
#            float_value = 5000.0
#        return float_value
#    ## SECOND TURN: stone has placed exactly one piece and the opponent has
#    ## placed exactly one -- reward landing in one of the four bands
#    ## surrounding the opponent's first stone. Spec was rows/cols in [3,6]
#    ## or [9,12] on a 15x15 board, scaled the same way.
#    if stone_count == 1 and opponent_count == 1:
#        lo1, hi1 = round(board_len * 3 / 15), round(board_len * 6 / 15)
#        lo2, hi2 = round(board_len * 9 / 15), round(board_len * 12 / 15)
#        for row_band in ((lo1, hi1), (lo2, hi2)):
#            for col_band in ((lo1, hi1), (lo2, hi2)):
#                if row_band[0] <= row_idx <= row_band[1] and col_band[0] <= col_idx <= col_band[1]:
#                    float_value = 5000.0
#        return float_value
#    return float_value
#    for row in iter_seqs(board):
#        for col in row():
#            return float_value
#    return float_value

def student_eval(board, stone):
    '''
In every single turn, we check if we have a1, and if we do, we act on it.

Order is determined by sum(attack) - sum(defend) = determinant
if determinant > 0, attack
if determinant < 0, defend

ATTACK: Value of a1 is 200,000
        Value of a2c, a2d, a2h, a2j is 100,000 if space after endpoint 'stone' is empty
            else, the value is 1
        Value of a2b, a2k is 100,000 if space after endpoint 'stone' is empty
            else, the value is 1
        Value of a2e, a2f, a2g, and a2i are 1
        '''
##Opening:
    # If we start eith zero stones in the board, then we apply our opening function to attack being stone X in the index of the #middle of the board, that is from [5,10] row and [5,10] column randomly. If we are O and start second in the board, and #there's one piece X in the board, then we decided to start between row index [3,6] and [9, 12] and for columns [3,6] and [9,12]. Therefore, after our first turn with the stone X, which start first on the board, we decide to place our stone diagonally, if the opponent O chose to place it sone diagonally from our stone X, we place our second stone in the diagonal opposite to the diagonal placed by the enemy, that is, for example, if the opponent placed the stone in a down diagonal from left to right to our stone, we place our second stone X in the up diagonal from left to right. If the opponent didn't place anything around our first placed stone, then we randomly choose either of the two diagonals to place our stone at.

    # If we are the ones that start first then we do this strategy, to second turn but if we are the second to start, then after our first turn, we leave the algorithm we wrote to do the rest
    #Attack values
    Win_val = 10000000.0
    Opp_Win_Val = -10000000.0
    A1_value = 1200000.0
    A1_gap_value = 1000000.0
    A2_value = 6000.0
    A2_gap_value = 4000.0
    A3_value = 60.0
    A3_gap_value = 40.0

    #Defend values
    D1_value = 800000.0
    D1_gap_value = 600000.0
    D2_value = 5000.0
    D2_gap_value = 3000.0
    D3_value = 50.0
    D3_gap_value = 30.0
    sum_attack = 0
    sum_defend = 0
    opponent_stone = other_stone(stone)

## STRATEGY PLAY FUNCTIONS
    # Attack function
    for i in iter_seqs(board):
        if consecutive_k(i, 5, stone) > 0:
            return Win_val
        # A1 priority
        sum_attack += opened_ends(i, 4, stone) * A1_value
        sum_attack += p_blocked_ends(i, 4, stone) * A1_value
        sum_attack += detect_split_k(i, 4, stone) * A1_gap_value
        # A2 priority
        sum_attack += opened_ends(i, 3, stone) * A2_value
        sum_attack += p_blocked_ends(i, 3, stone) * A2_value
        sum_attack += detect_split_k(i, 3, stone) * A2_gap_value
        # Rest of combinations
        sum_attack += opened_ends(i, 2, stone) * A3_value
        sum_attack += p_blocked_ends(i, 2, stone) * A3_value
        sum_attack += detect_split_k(i, 2, stone) * A3_gap_value
    #sum_defend = gomu_gomu_no_Mi(board, stone)
    sum_attack += gomu_gomu_no_Mi(board, stone)
        # Defend function
    for i in iter_seqs(board):
        if consecutive_k(i, 5, opponent_stone) > 0:
            return Opp_Win_Val
        # A1 priority
        sum_defend += opened_ends(i, 4, opponent_stone) * D1_value
        sum_defend += p_blocked_ends(i, 4, opponent_stone) * D1_value
        sum_defend += detect_split_k(i, 4, opponent_stone) * D1_gap_value
        # A2 priority
        sum_defend += opened_ends(i, 3, opponent_stone) * D2_value
        sum_defend += p_blocked_ends(i, 3, opponent_stone) * D2_value
        sum_defend += detect_split_k(i, 3, opponent_stone) * D2_gap_value
        # Rest of combinations
        sum_defend += opened_ends(i, 2, opponent_stone) * D3_value
        sum_defend += p_blocked_ends(i, 2, opponent_stone) * D3_value
        sum_defend += detect_split_k(i, 2, opponent_stone) * D3_gap_value
    sum_defend += gomu_gomu_no_Mi(board, opponent_stone)
    determine_move = sum_attack - sum_defend
    return determine_move


## Testing utilities
Your heuristic function will be plugged in to a `Strategy`, which uses a [game tree](https://en.wikipedia.org/wiki/Game_tree) to select which move to play. 

The following sections provide different ways to test and analyze your heuristic. You are more than welcome to create new tests or to copy any tweak any of the tests below. Just be sure to not modify the original ones so that you will always have them handy. 


### An empty starting board
The cell below is useful for seeing how your heuristic behaves on an empty starting board. You can also copy the cell and fill out the board to make your own custom starting board. 


In the example below, we see the following variables:
* `small_board`: A list of lists representing an 8x8 board.
* `difficulty_setting`: Used to set the difficulty of the opponent. `difficulty_setting` is the probability that the opponent will make an "intelligent" move, as opposed to a random move
* `slider_eval`: The heuristic function of your opponent.
* `p1` and `p2`: Yours and your opponent's respective strategies. You are player `X`.

Then, the `play_game` function simulates a game, using `small_board` as the starting point for `p1` and `p2` to play against eachother on.

In [7]:
# =============================================================================
# A playground to run your code against itself (or someone else)
# =============================================================================
from functools import partial

small_board = np.array([list(x) for x in """
-|-|-|-|-|-|-|-
-|-|-|-|-|-|-|-
-|-|-|-|-|-|-|-
-|-|-|-|-|-|-|-
-|-|-|-|-|-|-|-
-|-|-|-|-|-|-|-
-|-|-|-|-|-|-|-
-|-|-|-|-|-|-|-""".replace("|", "").split('\n')[1:]])

# The difficulty_setting variable can be anything in [0, 1], with
# higher values meaning a harder bot. 0 is the random bot
# and 1 is the "hardest" bot
difficulty_setting = 0.0
slider_eval = partial(kille.slider_eval, difficulty=difficulty_setting)

p1 = Strategy("X", student_eval, max_depth=1) # Do not change the search depths!
p2 = Strategy("O", slider_eval, max_depth=1)
play_game(small_board, p1, p2) # Start with the small board, it is better for debugging.

# 1 	Player X 	Move: 0,0
X|-|-|-|-|-|-|-
-|-|-|-|-|-|-|-
-|-|-|-|-|-|-|-
-|-|-|-|-|-|-|-
-|-|-|-|-|-|-|-
-|-|-|-|-|-|-|-
-|-|-|-|-|-|-|-
-|-|-|-|-|-|-|-

# 2 	Player O 	Move: 7,7
X|-|-|-|-|-|-|-
-|-|-|-|-|-|-|-
-|-|-|-|-|-|-|-
-|-|-|-|-|-|-|-
-|-|-|-|-|-|-|-
-|-|-|-|-|-|-|-
-|-|-|-|-|-|-|-
-|-|-|-|-|-|-|O

# 3 	Player X 	Move: 1,1
X|-|-|-|-|-|-|-
-|X|-|-|-|-|-|-
-|-|-|-|-|-|-|-
-|-|-|-|-|-|-|-
-|-|-|-|-|-|-|-
-|-|-|-|-|-|-|-
-|-|-|-|-|-|-|-
-|-|-|-|-|-|-|O

# 4 	Player O 	Move: 6,6
X|-|-|-|-|-|-|-
-|X|-|-|-|-|-|-
-|-|-|-|-|-|-|-
-|-|-|-|-|-|-|-
-|-|-|-|-|-|-|-
-|-|-|-|-|-|-|-
-|-|-|-|-|-|O|-
-|-|-|-|-|-|-|O

# 5 	Player X 	Move: 2,2
X|-|-|-|-|-|-|-
-|X|-|-|-|-|-|-
-|-|X|-|-|-|-|-
-|-|-|-|-|-|-|-
-|-|-|-|-|-|-|-
-|-|-|-|-|-|-|-
-|-|-|-|-|-|O|-
-|-|-|-|-|-|-|O

# 6 	Player O 	Move: 4,5
X|-|-|-|-|-|-|-
-|X|-|-|-|-|-|-
-|-|X|-|-|-|-|-
-|-|-|-|-|-|-|-
-|-|-|-|-|O|-|-
-|-|-|-|-|-|-|-
-|-|-|-|-|-|O|-
-|-|-|-|-|-|-|O

# 7 	Player X 	Move: 3,3
X|-|-|-|-|-|-|-
-|X|-|-|-|-|-|-
-|-|X|-|-|-|-|-
-|-

'X'

### The best defense...
Given the board below, where should player `X` place their stone if it is currently their turn?

When you execute the cell, your `student_eval` is used to report the "score" that your heuristic assigned to `test_board`. Then, a strategy is created using your heuristic, and the move chosen by your strategy is reported.

**Remember**, the move is selected by evaluating the game tree at the leaves. In this cell, the depth of the game tree is $1$. This means that the strategy will iterate through each of your possible moves, and for each one, pass the resulting board to `student_eval`.

In [8]:
# =============================================================================
# A playground to see what your code's next move would be and what the evaluation
# and/or the current position is
# =============================================================================

test_board = np.array([list(x) for x in """
O|X|X|X|-|-|X|-
-|O|-|-|-|-|-|-
-|-|O|-|-|-|-|-
-|-|-|O|-|-|-|-
-|-|-|-|-|-|-|-
-|-|-|-|-|-|-|-
-|-|-|-|-|-|-|-
-|-|-|-|-|-|-|-""".replace("|", "").split('\n')[1:]])

print("Board score", student_eval(test_board, "X"))

p1 = Strategy("X", student_eval, max_depth=1)
print("Next move", p1.get_move(test_board, "X"))

Board score -1594060.0
Next move (np.int64(4), np.int64(4))


### An easy win

In this test case, it should be clear that player `X` can get an easy victory, regardless of whose turn it is.

In [9]:
difficulty_setting = 1.0
slider_eval = partial(kille.slider_eval, difficulty=difficulty_setting)


# Test 1: Easy winning row. Your bot should win in one move.
small_board = np.array([list(x) for x in """
-|-|-|-|-|-|-|-
-|-|-|-|-|-|-|-
-|-|-|-|-|-|-|-
-|-|-|X|X|X|X|-
-|-|-|-|-|-|-|-
-|-|-|-|-|-|-|-
-|-|-|-|-|-|-|-
-|-|-|-|-|-|-|-""".replace("|", "").split('\n')[1:]])

p1 = Strategy("X", student_eval, max_depth=1) # Do not change the search depths!
p2 = Strategy("O", slider_eval, max_depth=1)
play_game(small_board, p1, p2)

# 1 	Player X 	Move: 3,2
-|-|-|-|-|-|-|-
-|-|-|-|-|-|-|-
-|-|-|-|-|-|-|-
-|-|X|X|X|X|X|-
-|-|-|-|-|-|-|-
-|-|-|-|-|-|-|-
-|-|-|-|-|-|-|-
-|-|-|-|-|-|-|-

Student won!


'X'

### Distractions
Another test case where there is an easy win, but some strategies may be tempted to try and block `O`...

In [10]:
# Test 2: This test is a bit harder.
difficulty_setting = 1.0
slider_eval = partial(kille.slider_eval, difficulty=difficulty_setting)
small_board = np.array([list(x) for x in """
-|-|-|-|-|-|-|-
-|-|-|-|-|-|-|-
-|-|-|-|-|-|-|-
-|-|-|X|X|X|X|-
-|-|-|-|-|-|-|-
-|-|-|O|O|O|O|-
-|-|-|-|-|-|-|-
-|-|-|-|-|-|-|-""".replace("|", "").split('\n')[1:]])

p1 = Strategy("X", student_eval, max_depth=1) # Do not change the search depths!
p2 = Strategy("O", slider_eval, max_depth=1)
play_game(small_board, p1, p2)


# 1 	Player X 	Move: 3,2
-|-|-|-|-|-|-|-
-|-|-|-|-|-|-|-
-|-|-|-|-|-|-|-
-|-|X|X|X|X|X|-
-|-|-|-|-|-|-|-
-|-|-|O|O|O|O|-
-|-|-|-|-|-|-|-
-|-|-|-|-|-|-|-

Student won!


'X'

### Finding winning patterns
The following board is "won" for player `X`, i.e., if `X` plays perfectly, there is no way `O` can prevent them from winning.Can you see how? Can your strategy?

In [11]:
difficulty_setting = 1
slider_eval = partial(kille.slider_eval, difficulty=difficulty_setting)
small_board = np.array([list(x) for x in """
-|-|-|-|-|-|-|-
-|-|-|-|-|-|-|-
-|-|X|-|-|-|-|-
-|-|-|X|O|-|-|-
-|-|-|O|X|-|-|-
-|-|-|-|-|-|-|-
-|-|-|-|-|-|-|-
-|-|-|-|-|-|-|-""".replace("|", "").split('\n')[1:]])

p1 = Strategy("X", student_eval, max_depth=1) # Do not change the search depths!
p2 = Strategy("O", slider_eval, max_depth=1)
play_game(small_board, p1, p2)

# 1 	Player X 	Move: 5,5
-|-|-|-|-|-|-|-
-|-|-|-|-|-|-|-
-|-|X|-|-|-|-|-
-|-|-|X|O|-|-|-
-|-|-|O|X|-|-|-
-|-|-|-|-|X|-|-
-|-|-|-|-|-|-|-
-|-|-|-|-|-|-|-

# 2 	Player O 	Move: 1,1
-|-|-|-|-|-|-|-
-|O|-|-|-|-|-|-
-|-|X|-|-|-|-|-
-|-|-|X|O|-|-|-
-|-|-|O|X|-|-|-
-|-|-|-|-|X|-|-
-|-|-|-|-|-|-|-
-|-|-|-|-|-|-|-

# 3 	Player X 	Move: 6,6
-|-|-|-|-|-|-|-
-|O|-|-|-|-|-|-
-|-|X|-|-|-|-|-
-|-|-|X|O|-|-|-
-|-|-|O|X|-|-|-
-|-|-|-|-|X|-|-
-|-|-|-|-|-|X|-
-|-|-|-|-|-|-|-

Student won!


'X'

### Priorities
The following board is "won" for player `X` again. That being said, `O` is also one move away from winning...

In [12]:
difficulty_setting = 1
slider_eval = partial(kille.slider_eval, difficulty=difficulty_setting)
small_board = np.array([list(x) for x in """
-|O|-|-|-|-|-|-
-|-|-|-|O|-|-|-
-|X|-|-|-|O|-|-
-|-|X|-|O|X|-|-
-|-|-|O|-|-|X|-
-|-|O|-|X|-|-|-
-|X|-|-|-|-|-|-
-|-|-|-|-|-|-|-""".replace("|", "").split('\n')[1:]])

p1 = Strategy("X", student_eval, max_depth=1) # Change this to the slider eval to see how it should play out
p2 = Strategy("O", slider_eval, max_depth=1)
play_game(small_board, p1, p2)

# 1 	Player X 	Move: 1,6
-|O|-|-|-|-|-|-
-|-|-|-|O|-|X|-
-|X|-|-|-|O|-|-
-|-|X|-|O|X|-|-
-|-|-|O|-|-|X|-
-|-|O|-|X|-|-|-
-|X|-|-|-|-|-|-
-|-|-|-|-|-|-|-

# 2 	Player O 	Move: 2,4
-|O|-|-|-|-|-|-
-|-|-|-|O|-|X|-
-|X|-|-|O|O|-|-
-|-|X|-|O|X|-|-
-|-|-|O|-|-|X|-
-|-|O|-|X|-|-|-
-|X|-|-|-|-|-|-
-|-|-|-|-|-|-|-

# 3 	Player X 	Move: 2,6
-|O|-|-|-|-|-|-
-|-|-|-|O|-|X|-
-|X|-|-|O|O|X|-
-|-|X|-|O|X|-|-
-|-|-|O|-|-|X|-
-|-|O|-|X|-|-|-
-|X|-|-|-|-|-|-
-|-|-|-|-|-|-|-

# 4 	Player O 	Move: 4,4
-|O|-|-|-|-|-|-
-|-|-|-|O|-|X|-
-|X|-|-|O|O|X|-
-|-|X|-|O|X|-|-
-|-|-|O|O|-|X|-
-|-|O|-|X|-|-|-
-|X|-|-|-|-|-|-
-|-|-|-|-|-|-|-

# 5 	Player X 	Move: 0,4
-|O|-|-|X|-|-|-
-|-|-|-|O|-|X|-
-|X|-|-|O|O|X|-
-|-|X|-|O|X|-|-
-|-|-|O|O|-|X|-
-|-|O|-|X|-|-|-
-|X|-|-|-|-|-|-
-|-|-|-|-|-|-|-

# 6 	Player O 	Move: 4,2
-|O|-|-|X|-|-|-
-|-|-|-|O|-|X|-
-|X|-|-|O|O|X|-
-|-|X|-|O|X|-|-
-|-|O|O|O|-|X|-
-|-|O|-|X|-|-|-
-|X|-|-|-|-|-|-
-|-|-|-|-|-|-|-

# 7 	Player X 	Move: 3,6
-|O|-|-|X|-|-|-
-|-|-|-|O|-|X|-
-|X|-|-|O|O|X|-
-|-

'X'

f### A real game
The following board can help show how the strategy works when there are already some stones in play.

In [13]:
# Test 3: A test for playing from the starting position
difficulty_setting = 0.25
slider_eval = partial(kille.slider_eval, difficulty=difficulty_setting)
small_board = np.array([list(x) for x in """
-|-|-|-|-|-|-|-
-|-|-|-|-|-|-|-
-|-|-|-|-|-|-|-
-|-|-|X|O|-|-|-
-|-|-|O|X|-|-|-
-|-|-|-|-|-|-|-
-|-|-|-|-|-|-|-
-|-|-|-|-|-|-|-""".replace("|", "").split('\n')[1:]])

p1 = Strategy("X", student_eval, max_depth=1) # Do not change the search depths!
p2 = Strategy("O", slider_eval, max_depth=1)
play_game(small_board, p1, p2)

# 1 	Player X 	Move: 2,2
-|-|-|-|-|-|-|-
-|-|-|-|-|-|-|-
-|-|X|-|-|-|-|-
-|-|-|X|O|-|-|-
-|-|-|O|X|-|-|-
-|-|-|-|-|-|-|-
-|-|-|-|-|-|-|-
-|-|-|-|-|-|-|-

# 2 	Player O 	Move: 7,2
-|-|-|-|-|-|-|-
-|-|-|-|-|-|-|-
-|-|X|-|-|-|-|-
-|-|-|X|O|-|-|-
-|-|-|O|X|-|-|-
-|-|-|-|-|-|-|-
-|-|-|-|-|-|-|-
-|-|O|-|-|-|-|-

# 3 	Player X 	Move: 5,5
-|-|-|-|-|-|-|-
-|-|-|-|-|-|-|-
-|-|X|-|-|-|-|-
-|-|-|X|O|-|-|-
-|-|-|O|X|-|-|-
-|-|-|-|-|X|-|-
-|-|-|-|-|-|-|-
-|-|O|-|-|-|-|-

# 4 	Player O 	Move: 6,5
-|-|-|-|-|-|-|-
-|-|-|-|-|-|-|-
-|-|X|-|-|-|-|-
-|-|-|X|O|-|-|-
-|-|-|O|X|-|-|-
-|-|-|-|-|X|-|-
-|-|-|-|-|O|-|-
-|-|O|-|-|-|-|-

# 5 	Player X 	Move: 1,1
-|-|-|-|-|-|-|-
-|X|-|-|-|-|-|-
-|-|X|-|-|-|-|-
-|-|-|X|O|-|-|-
-|-|-|O|X|-|-|-
-|-|-|-|-|X|-|-
-|-|-|-|-|O|-|-
-|-|O|-|-|-|-|-

Student won!


'X'

## Running a tournament
The following cell will run two tournaments, one on an empty starting board, and another on random starting boards. For each level of difficulty, `N=10` games are played between your strategy and the corresponding bot. 

In [86]:
# =============================================================================
# Tournament code! This code will run your function against bots of different
# difficulty.
# =============================================================================

N = 20
# Number of games per matchup

easy_bot = partial(kille.slider_eval, difficulty=0)
medium_bot = partial(kille.slider_eval, difficulty=0.2)
hard_bot = partial(kille.slider_eval, difficulty=0.5)
hardest_bot = partial(kille.slider_eval, difficulty=1.0)

eval_functions = [
    (easy_bot, "easy bot"),
    (medium_bot, "medium bot"),
    (hard_bot, "hard bot"),
    (hardest_bot, "hardest bot"),
    (student_eval, "My eval")
]

print("Empty start")
run_tournament(eval_functions, num_trials=N, board_size=8, num_random_moves=0, n_cpu=1)

print("\n\nRandom start")
run_tournament(eval_functions, num_trials=N, board_size=8, num_random_moves=-1, n_cpu=1)

Empty start


  0%|          | 0/4 [00:00<?, ?it/s]


+---------+----------+------------+----------+-------------+---------+
|    20   | easy bot | medium bot | hard bot | hardest bot | My eval |
+---------+----------+------------+----------+-------------+---------+
| My eval |    20    |     20     |    16    |      20     |    -    |
+---------+----------+------------+----------+-------------+---------+


Random start


  0%|          | 0/4 [00:00<?, ?it/s]


+---------+----------+------------+----------+-------------+---------+
|    20   | easy bot | medium bot | hard bot | hardest bot | My eval |
+---------+----------+------------+----------+-------------+---------+
| My eval |    20    |     19     |    12    |      5      |    -    |
+---------+----------+------------+----------+-------------+---------+


Copyright ©2024 Bryce Kille (blk6@rice.edu), Risa B Myers  (rbm2@rice.edu)

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.